# Consumer Shopping Behavior Survey — Portfolio EDA

**Dataset:** 500 survey responses across 25+ countries and 30 variables.

## Objective
Turn the survey data into **decision-oriented analysis** for a growth, product, or marketing team.

### Business questions
1. Who are the respondents?
2. How do they shop?
3. What influences purchase decisions?
4. What drives spending and payment behavior?
5. What is associated with satisfaction and purchase regret?
6. Which customer groups show distinct shopping behaviors?
7. What actions should a business prioritize?

> **Analysis principle:** descriptive statistics → relationships → statistical evidence → customer segments → business recommendations.


In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 110,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
})

DATA_PATH = Path("consumer_shopping_behavior_survey.csv")

if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    if candidates:
        DATA_PATH = candidates[0]
    else:
        raise FileNotFoundError(
            "CSV not found. Put consumer_shopping_behavior_survey.csv "
            "in the same folder as this notebook."
        )

df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()


## 1. Data Quality & Preparation

Survey exports contain skipped questions and conditional fields. We therefore **measure missingness first**, preserve valid observations, and only transform fields when the transformation is justified.


In [ ]:
# Structural overview
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)

display(quality)

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Response_IDs:", df["Response_ID"].duplicated().sum())

# Convert obvious numeric fields
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce", dayfirst=True)

numeric_cols = [
    "Age",
    "Review_Influence_Score",
    "Social_Ads_Influence_Score",
    "Price_Comparison_Frequency",
    "Discount_Importance_Score",
    "Likely_To_Buy_After_Ad_Score",
    "Impulse_Purchase_Frequency",
    "Online_Shopping_Satisfaction_Score",
    "Trust_In_Online_Reviews_Score",
    "Regret_After_Purchase_Frequency",
]

numeric_cols = [c for c in numeric_cols if c in df.columns]

print("\nNumeric summary:")
display(df[numeric_cols].describe().T)


In [ ]:
# Missingness visualization
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots()
missing_pct.plot(kind="bar", ax=ax)
ax.set_title("Missing Data by Variable")
ax.set_ylabel("Missing (%)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=75)
plt.tight_layout()
plt.show()


### Data-quality interpretation

Do not automatically drop all rows with missing values. Several fields are conditional — for example, a respondent may not provide an online-shopping reason if they primarily shop in-store.

For modeling, missing values should be handled **feature-by-feature** rather than with a blanket row deletion.


## 2. Executive Dashboard

The first analytical view should answer: **What does this sample look like at a glance?**


In [ ]:
# Core KPIs
kpis = {
    "Respondents": len(df),
    "Median age": df["Age"].median(),
    "Online best mode (%)": (df["Best_Shopping_Mode"].eq("Online").mean() * 100),
    "Mobile users (%)": (df["Primary_Device"].eq("Smartphone").mean() * 100),
    "Mean satisfaction": df["Online_Shopping_Satisfaction_Score"].mean(),
    "Mean review trust": df["Trust_In_Online_Reviews_Score"].mean(),
}

for k, v in kpis.items():
    print(f"{k}: {v:.1f}" if isinstance(v, (float, np.floating)) else f"{k}: {v}")


## 3. Who Are the Respondents?

Profile the population before interpreting behavioral patterns. Demographics provide context but should not be treated as causal explanations.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df["Age"].dropna().plot(
    kind="hist", bins=15, ax=axes[0], edgecolor="black"
)
axes[0].axvline(df["Age"].median(), linestyle="--", label="Median")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")
axes[0].legend()

occupation = df["Occupation_Status"].value_counts().head(8)
occupation.sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Top Occupation Groups")
axes[1].set_xlabel("Respondents")

plt.tight_layout()
plt.show()


In [ ]:
# Income and geography
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

income_order = [
    "Under $1,000", "$1,000-3,000", "$3,000-5,000",
    "$5,000-10,000", "Over $10,000"
]
income = df["Monthly_Income_Range"].value_counts()
income = income.reindex([x for x in income_order if x in income.index]).dropna()
income.plot(kind="bar", ax=axes[0])
axes[0].set_title("Monthly Income Range")
axes[0].set_ylabel("Respondents")
axes[0].tick_params(axis="x", rotation=35)

country = df["Country_Region"].value_counts().head(12).sort_values()
country.plot(kind="barh", ax=axes[1])
axes[1].set_title("Top Countries / Regions")
axes[1].set_xlabel("Respondents")

plt.tight_layout()
plt.show()


## 4. Shopping Channel, Platform & Device

The key question is not simply which channel is most popular, but whether **channel, platform, and device behavior align**.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, title in [
    (axes[0,0], "Best_Shopping_Mode", "Preferred Shopping Mode"),
    (axes[0,1], "Shop_Most_Where", "Where People Shop Most"),
    (axes[1,0], "Preferred_Platform", "Preferred Platform"),
    (axes[1,1], "Primary_Device", "Primary Device"),
]:
    counts = df[col].value_counts().head(10).sort_values()
    counts.plot(kind="barh", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Respondents")

plt.tight_layout()
plt.show()


In [ ]:
# Shopping frequency
freq_order = ["Never", "Rarely", "Occasionally", "Often", "Always"]

freq = df["Online_Shopping_Frequency"].value_counts()
freq = freq.reindex([x for x in freq_order if x in freq.index]).dropna()

fig, ax = plt.subplots()
freq.plot(kind="bar", ax=ax)
ax.set_title("Online Shopping Frequency")
ax.set_ylabel("Respondents")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


## 5. Purchase Drivers

Compare reviews, social advertising, price comparison, discounts, and ad responsiveness on a common scale.

Because these are survey scores, the result should be interpreted as **reported influence**, not observed causal impact.


In [ ]:
influence_cols = [
    "Review_Influence_Score",
    "Social_Ads_Influence_Score",
    "Price_Comparison_Frequency",
    "Discount_Importance_Score",
    "Likely_To_Buy_After_Ad_Score",
]

influence_summary = (
    df[influence_cols]
    .agg(["mean", "median", "count"])
    .T
    .sort_values("mean", ascending=False)
)

display(influence_summary)

fig, ax = plt.subplots(figsize=(10, 5))
influence_summary["mean"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Average Purchase-Influence Scores")
ax.set_xlabel("Mean score")
plt.tight_layout()
plt.show()


In [ ]:
# Planned vs impulse behavior
list_map = {
    "Never": 0,
    "Sometimes": 1,
    "Always": 2
}

planned = df["Makes_Shopping_List"].map(list_map)
impulse = df["Impulse_Purchase_Frequency"]

print("Mean impulse score by shopping-list behavior:")
display(
    df.assign(Shopping_List_Score=planned)
      .groupby("Shopping_List_Score")["Impulse_Purchase_Frequency"]
      .agg(["count", "mean", "median"])
)

rho, p = stats.spearmanr(
    planned.dropna(),
    impulse.loc[planned.dropna().index].dropna()
)
print(f"Spearman correlation (list planning vs impulse): rho={rho:.3f}, p={p:.4f}")


## 6. Spending, Payment & BNPL

Analyze financial behavior without assuming that payment method causes spending differences.


In [ ]:
spend_order = ["Under $50", "$50-150", "$150-300", "$300-500", "$500+"]

spend = df["Avg_Monthly_Spend_NonEssentials"].value_counts()
spend = spend.reindex([x for x in spend_order if x in spend.index]).dropna()

fig, ax = plt.subplots()
spend.plot(kind="bar", ax=ax)
ax.set_title("Average Monthly Non-Essential Spending")
ax.set_ylabel("Respondents")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

payment = df["Preferred_Payment_Method"].value_counts().sort_values()
fig, ax = plt.subplots()
payment.plot(kind="barh", ax=ax)
ax.set_title("Preferred Payment Method")
ax.set_xlabel("Respondents")
plt.tight_layout()
plt.show()


In [ ]:
bnpl = df["Uses_BNPL_Installments"].value_counts().sort_values()

fig, ax = plt.subplots()
bnpl.plot(kind="barh", ax=ax)
ax.set_title("BNPL / Installment Usage")
ax.set_xlabel("Respondents")
plt.tight_layout()
plt.show()

bnpl_users = ~df["Uses_BNPL_Installments"].fillna("No, never").eq("No, never")
print(f"Share using BNPL at least occasionally: {bnpl_users.mean()*100:.1f}%")


## 7. Satisfaction & Regret

A useful product question is whether satisfaction is associated with shopping mode and whether regret is associated with impulse behavior.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df["Online_Shopping_Satisfaction_Score"].dropna().plot(
    kind="hist", bins=10, ax=axes[0], edgecolor="black"
)
axes[0].set_title("Online Shopping Satisfaction")
axes[0].set_xlabel("Satisfaction score")

df["Regret_After_Purchase_Frequency"].dropna().plot(
    kind="hist", bins=10, ax=axes[1], edgecolor="black"
)
axes[1].set_title("Post-Purchase Regret")
axes[1].set_xlabel("Regret frequency")

plt.tight_layout()
plt.show()


In [ ]:
# Satisfaction by preferred shopping mode
satisfaction_by_mode = (
    df.groupby("Best_Shopping_Mode")["Online_Shopping_Satisfaction_Score"]
      .agg(["count", "mean", "median", "std"])
      .sort_values("mean", ascending=False)
)

display(satisfaction_by_mode)

# Impulse vs regret
paired = df[["Impulse_Purchase_Frequency", "Regret_After_Purchase_Frequency"]].dropna()

rho, p = stats.spearmanr(
    paired["Impulse_Purchase_Frequency"],
    paired["Regret_After_Purchase_Frequency"]
)

print(f"Impulse vs regret: Spearman rho={rho:.3f}, p={p:.4f}")


## 8. Relationship Analysis

Correlation alone is not enough. We combine:
- Spearman correlations for ordinal scores
- contingency tables for categorical relationships
- chi-square tests for categorical association

A statistically significant result does **not** automatically imply a practically important relationship.


In [ ]:
corr_cols = [
    "Review_Influence_Score",
    "Social_Ads_Influence_Score",
    "Price_Comparison_Frequency",
    "Discount_Importance_Score",
    "Likely_To_Buy_After_Ad_Score",
    "Impulse_Purchase_Frequency",
    "Online_Shopping_Satisfaction_Score",
    "Trust_In_Online_Reviews_Score",
    "Regret_After_Purchase_Frequency",
]

corr_cols = [c for c in corr_cols if c in df.columns]

corr = df[corr_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=75, ha="right")
ax.set_yticklabels(corr.columns)
ax.set_title("Spearman Correlation Matrix")
plt.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()


In [ ]:
# Identify the strongest relationships
pairs = []
for i, a in enumerate(corr.columns):
    for b in corr.columns[i+1:]:
        pairs.append((a, b, corr.loc[a, b], abs(corr.loc[a, b])))

strongest = (
    pd.DataFrame(pairs, columns=["Variable_1", "Variable_2", "rho", "abs_rho"])
      .sort_values("abs_rho", ascending=False)
      .head(10)
)

display(strongest)


In [ ]:
# Categorical association: preferred platform vs best shopping mode
ct = pd.crosstab(
    df["Preferred_Platform"],
    df["Best_Shopping_Mode"]
)

display(ct)

chi2, p, dof, expected = stats.chi2_contingency(ct)

n = ct.to_numpy().sum()
r, k = ct.shape
cramers_v = np.sqrt((chi2 / n) / min(k - 1, r - 1))

print(f"Chi-square={chi2:.2f}, dof={dof}, p={p:.4f}")
print(f"Cramer's V={cramers_v:.3f}")


## 9. Income vs Discount Sensitivity

Rather than relying on a visual impression, test whether discount-importance scores differ across income groups.

Use Kruskal–Wallis because income is represented as ordered categories and the influence score is ordinal.


In [ ]:
groups = []
labels = []

for label in income_order:
    if label in df["Monthly_Income_Range"].dropna().unique():
        values = df.loc[
            df["Monthly_Income_Range"].eq(label),
            "Discount_Importance_Score"
        ].dropna()
        if len(values) > 0:
            groups.append(values)
            labels.append(label)

if len(groups) >= 2:
    h, p = stats.kruskal(*groups)
    print(f"Kruskal-Wallis H={h:.2f}, p={p:.4f}")

    medians = (
        df.groupby("Monthly_Income_Range")["Discount_Importance_Score"]
          .agg(["count", "mean", "median"])
          .reindex(income_order)
    )
    display(medians)


## 10. Customer Segmentation

Create interpretable shopper profiles using behavioral variables rather than demographics alone.

The clustering is exploratory. Cluster labels are assigned **after** inspecting the cluster characteristics.


In [ ]:
cluster_features = [
    "Review_Influence_Score",
    "Social_Ads_Influence_Score",
    "Price_Comparison_Frequency",
    "Discount_Importance_Score",
    "Likely_To_Buy_After_Ad_Score",
    "Impulse_Purchase_Frequency",
    "Online_Shopping_Satisfaction_Score",
    "Trust_In_Online_Reviews_Score",
    "Regret_After_Purchase_Frequency",
]

cluster_data = df[cluster_features].dropna().copy()

scaler = StandardScaler()
X = scaler.fit_transform(cluster_data)

scores = []
models = {}

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X)
    sil = silhouette_score(X, labels)
    scores.append((k, sil))
    models[k] = model

silhouette_df = pd.DataFrame(scores, columns=["k", "silhouette_score"])
display(silhouette_df)

best_k = silhouette_df.loc[
    silhouette_df["silhouette_score"].idxmax(), "k"
]

print("Selected k:", int(best_k))


In [ ]:
best_model = models[int(best_k)]
cluster_data = cluster_data.copy()
cluster_data["Cluster"] = best_model.labels_

profile = cluster_data.groupby("Cluster")[cluster_features].mean().round(2)
display(profile)

fig, ax = plt.subplots(figsize=(12, 6))
profile.T.plot(kind="bar", ax=ax)
ax.set_title("Behavioral Profile of Customer Clusters")
ax.set_ylabel("Average score")
ax.tick_params(axis="x", rotation=65)
plt.tight_layout()
plt.show()


### How to interpret clusters

Do **not** call a cluster "high-value", "loyal", or "impulsive" simply because it has a high value on one variable.

Instead:
1. inspect its full profile,
2. compare it with the other clusters,
3. identify the dominant behavior pattern,
4. give it a business-friendly name only after that inspection.


## 11. Business Insights

Use this section as the bridge between analysis and decision-making.

### Insight framework

For each important result, write:

**Finding → Evidence → Business implication → Recommended action**

Example:

> **Finding:** Mobile devices represent a large share of the sample.  
> **Evidence:** Compare the smartphone share with other devices.  
> **Implication:** Mobile UX may affect a substantial portion of the customer journey.  
> **Action:** Prioritize mobile checkout speed, product discovery, and review visibility.

Avoid presenting a correlation as proof of causation.


In [ ]:
# Automatically generate a few evidence tables for the final discussion

print("Top spending categories:")
display(df["Top_Spending_Category"].value_counts().head(10).to_frame("respondents"))

print("Shopping mode:")
display(
    df["Best_Shopping_Mode"]
      .value_counts(normalize=True)
      .mul(100)
      .round(1)
      .to_frame("share_pct")
)

print("Primary device:")
display(
    df["Primary_Device"]
      .value_counts(normalize=True)
      .mul(100)
      .round(1)
      .to_frame("share_pct")
)


## 12. Final Takeaways

### What this notebook now demonstrates

- **Data quality:** missingness, duplicates, types, and descriptive statistics
- **EDA:** demographic and behavioral distributions
- **Relationship analysis:** correlations and cross-tabulation
- **Statistical testing:** Spearman, chi-square, Cramer's V, and Kruskal–Wallis
- **Segmentation:** K-Means + silhouette-based model selection
- **Business thinking:** translating evidence into actions

### Limitations

- The dataset is a survey sample, so results may not represent the broader population.
- Self-reported behavior can differ from actual transaction behavior.
- Cross-sectional survey data cannot establish causality.
- Missing values may be systematic because some questions are conditional.
- Segment names should be treated as analytical interpretations, not ground truth.

### Next steps

1. Add transaction-level data if available.
2. Build a predictive model for purchase propensity or satisfaction.
3. Validate the segments on a larger sample.
4. Compare behavioral patterns across countries or demographic groups.
5. Build a Power BI/Tableau dashboard from the cleaned analytical dataset.


# Portfolio Summary

**Project title:** Consumer Shopping Behavior — EDA & Customer Segmentation

**Tools:** Python, Pandas, NumPy, Matplotlib, SciPy, Scikit-learn

**Key skills demonstrated:** data cleaning, missing-data analysis, exploratory analysis, statistical testing, correlation analysis, categorical association, customer segmentation, and business recommendations.

> This notebook is designed to show not only that you can make charts, but that you can use data to answer business questions and communicate decisions.
